# Symbolic nonlinear observability

In [10]:
import numpy as np
import sympy as sp
from IPython.display import display

In [11]:
# Import functions directly from github
# Important: note that we use raw.githubusercontent.com, not github.com

import requests
url = 'https://raw.githubusercontent.com/florisvb/Nonlinear_and_Data_Driven_Estimation/main/Utility/symbolic_derivatives.py'
r = requests.get(url)

# Store the file to the colab working directory
with open('symbolic_derivatives.py', 'w') as f:
    f.write(r.text)

# import the function we want from that file
import symbolic_derivatives

# Example: downward facing constant altitude monocular camera

Here we have a single camera pointed down, moving laterally at constant altitude.

$
\mathbf{\dot{x}} = \mathbf{f}(\mathbf{x},\mathbf{u}) =
\frac{d}{dt}
\begin{bmatrix}
\bbox[yellow]{g} \\[0.3em]
\bbox[yellow]{z} \\[0.3em]
\end{bmatrix} =
\overset{f_0}{\begin{bmatrix}
0 \\[0.3em]
0
\end{bmatrix}} +
\overset{f_1}{\begin{bmatrix}
1 \\[0.3em]
0
\end{bmatrix}} \bbox[lightgreen]{u}
$

We have 1 measurement, the ventral optic flow. We can also assume we have lateral acceleration measurements, but in this case the acceleration is entirely defined by the control inputs we don't have to add it explicitly.

$
\mathbf{y} = \mathbf{{h}}(\mathbf{{x}}, \mathbf{{u}}) =
\begin{bmatrix}
\bbox[yellow]{g/z} \\[0.3em]
\end{bmatrix}
$

# Define states and dynamics in control affine form

In [12]:
S, V, I, R, beta, sigma = sp.symbols(['S', 'V', 'I', 'R', 'beta', 'sigma'])
x = [S, V, I, R, beta, sigma]

# Parameters
Lambda, mu, gamma = sp.symbols(['Lambda', 'mu', 'gamma'])

f_0 = sp.Matrix([
    Lambda - beta*S*I - mu*S,                    # Ṡ when α=0, κ=0
    -sigma*beta*V*I - mu*V,                      # V̇ when α=0, κ=0
    beta*S*I + sigma*beta*V*I - gamma*I - mu*I,  # İ when α=0, κ=0
    gamma*I - mu*R,                              # Ṙ when α=0, κ=0
    0,                                           # β̇ = 0
    0                                            # σ̇ = 0
])

f_1 = sp.Matrix([
    -S,      # this gets multiplied by α
    S,
    0,
    0,
    0,
    0
])

f_2 = sp.Matrix([
    beta*S*I,                       # this gets multiplied by κ
    sigma*beta*V*I,
    -beta*S*I - sigma*beta*V*I,
    0,
    0,
    0
])

# Define measurements

In [13]:
# Measurement function
N = sp.symbols('N')  # Total population (constant)

h = sp.Matrix([
    0.6*I,    # y₁: 60% of infected cases (under-reporting)
    V/N       # y₂: Vaccination coverage ratio
])

# Calculate each term in G

$G = [h, L_{f_0}h, L_{f1}h]$

In [14]:
# Calculate each term in G
# G = [h, L_{f_0}h, L_{f_1}h, L_{f_2}h]

# Take the derivative of h with respect to x along the vector f_0
L_f0_h = symbolic_derivatives.directional_derivative(h, x, f_0)
display(L_f0_h)

print('')

# Take the derivative of h with respect to x along the vector f_1
L_f1_h = symbolic_derivatives.directional_derivative(h, x, f_1)
display(L_f1_h)

print('')

# Take the derivative of h with respect to x along the vector f_2
L_f2_h = symbolic_derivatives.directional_derivative(h, x, f_2)
display(L_f2_h)

Matrix([
[0.6*I*S*beta + 0.6*I*V*beta*sigma - 0.6*I*gamma - 0.6*I*mu],
[                                (-I*V*beta*sigma - V*mu)/N]])

Matrix([
[  0],
[S/N]])

Matrix([
[-0.6*I*S*beta - 0.6*I*V*beta*sigma],
[                  I*V*beta*sigma/N]])

# Assemble G, take Jacobian

In [15]:
# Assemble G, take Jacobian
G = sp.Matrix([h, L_f0_h, L_f1_h, L_f2_h])
display(G)

Matrix([
[                                                     0.6*I],
[                                                       V/N],
[0.6*I*S*beta + 0.6*I*V*beta*sigma - 0.6*I*gamma - 0.6*I*mu],
[                                (-I*V*beta*sigma - V*mu)/N],
[                                                         0],
[                                                       S/N],
[                        -0.6*I*S*beta - 0.6*I*V*beta*sigma],
[                                          I*V*beta*sigma/N]])

In [16]:
# Jacobian of G with respect to states
display(G.jacobian(x))

Matrix([
[          0,                      0,                                                0.6, 0,                        0,             0],
[          0,                    1/N,                                                  0, 0,                        0,             0],
[ 0.6*I*beta,       0.6*I*beta*sigma, 0.6*S*beta + 0.6*V*beta*sigma - 0.6*gamma - 0.6*mu, 0,  0.6*I*S + 0.6*I*V*sigma,  0.6*I*V*beta],
[          0, (-I*beta*sigma - mu)/N,                                    -V*beta*sigma/N, 0,             -I*V*sigma/N,   -I*V*beta/N],
[          0,                      0,                                                  0, 0,                        0,             0],
[        1/N,                      0,                                                  0, 0,                        0,             0],
[-0.6*I*beta,      -0.6*I*beta*sigma,                     -0.6*S*beta - 0.6*V*beta*sigma, 0, -0.6*I*S - 0.6*I*V*sigma, -0.6*I*V*beta],
[          0,         I*beta*sigma/N,         

# Check the rank of G for a given operating point, $x_0$

In [17]:
# Check the rank of G for a given operating point, x_0
# Using realistic values from Nigeria TB data

x0 = {
    S: 60000000,      # Susceptible population
    V: 158430000,     # Vaccinated (71% of 223M)
    I: 361000,        # Infected (2023 reported cases)
    R: 12000000,      # Removed (cumulative recoveries)
    beta: 0.3,        # Transmission rate
    sigma: 0.8,       # Vaccine inefficiency
    Lambda: 9.04e-5,  # Recruitment rate
    mu: 4.3e-5,       # Natural mortality rate
    gamma: 1/180,     # Recovery rate (6 months treatment)
    N: 223000000      # Total population
}

display(G.jacobian(x).subs(x0))

print('')
print('Rank of G:')
G.jacobian(x).subs(x0).rank()

Matrix([
[          0,                     0,                0.6, 0,                 0,                 0],
[          0,           1/223000000,                  0, 0,                 0,                 0],
[    64980.0,               51984.0,   33613919.9966409, 0,  40448750400000.0,  10294781400000.0],
[          0, -0.000388520179565022, -0.170507623318386, 0, -205177.506726457, -76941.5650224215],
[          0,                     0,                  0, 0,                 0,                 0],
[1/223000000,                     0,                  0, 0,                 0,                 0],
[   -64980.0,              -51984.0,        -33613920.0, 0, -40448750400000.0, -10294781400000.0],
[          0,  0.000388520179372197,  0.170507623318386, 0,  205177.506726457,  76941.5650224215]])


Rank of G:


5

# Shortcut function to get G:

### First derivatives

In [18]:
# First derivatives
G1 = symbolic_derivatives.get_bigO(h, x, [f_0, f_1, f_2])

# Second derivatives
G2 = symbolic_derivatives.get_bigO(sp.Matrix.vstack(*G1), x, [f_0, f_1, f_2])

# Both first and second derivatives
G = sp.Matrix.vstack(*G1, *G2)
display(G)

Matrix([
[                                                                                                                                                                                 0.6*I],
[                                                                                                                                                                                   V/N],
[                                                                                                                            0.6*I*S*beta + 0.6*I*V*beta*sigma - 0.6*I*gamma - 0.6*I*mu],
[                                                                                                                                                            (-I*V*beta*sigma - V*mu)/N],
[                                                                                                                                                                                     0],
[                                                            

# Exercises:

1. Is the system observable with no controls (i.e. $u=0$)?
2. Is the system observable with control? (i.e. $u\neq0$)?
3. How many derivatives are needed, 1 or 2?
3. Apply the symbolic approach to the planar drone example with the measurements below. What is necessary in order for $z$ to be observable?